# NewsBot 2.0 · Notebook 04 – Language Understanding & Generation
**ITAI 2373 Final Project · Mourinho Hernanto**

**Goal:** summarise articles (extractive + abstractive), generate headlines, search by meaning, expand queries and enrich articles with context.

**Course modules:** 10 (neural LMs), 11 (generation), 12 (NLU)

> Runs top-to-bottom in Google Colab (first cell installs everything). All logic lives in `src/` –
> this notebook demonstrates and evaluates it.

In [1]:
# ⚙️ Setup – works in Google Colab (clones the repo + installs) and locally
import os, sys, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/ITAI2373-NewsBot-Final/ITAI2373-NewsBot-Final") if IN_COLAB else Path.cwd().resolve().parent
if IN_COLAB and not ROOT.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Haryapandhita/ITAI2373-NewsBot-Final.git",
                    "/content/ITAI2373-NewsBot-Final"], check=True)
sys.path.insert(0, str(ROOT))
from scripts.colab_setup import setup
ROOT = setup()

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)
# Transformer models (summarisation, translation, multilingual embeddings) are used when
# available (Colab: Runtime -> Change runtime type -> T4 GPU recommended); otherwise
# NewsBot falls back to classical methods automatically.
USE_TRANSFORMERS = os.environ.get("NEWSBOT_USE_TRANSFORMERS", "1") == "1"
print("Transformers enabled:", USE_TRANSFORMERS)

Project root: /home/claude/NewsBot-Final/ITAI2373-NewsBot-Final  |  Colab: False


Transformers enabled: True


In [2]:
from src.data_processing.data_loader import load_all
data = load_all(); bbc, webz = data["bbc"], data["webz"]
art = bbc[bbc.category == "business"].iloc[3]
print(art.title); print(art.text[:800], "...")

High fuel prices hit BA's profits
British Airways has blamed high fuel prices for a 40% drop in profits.
Reporting its results for the three months to 31 December 2004, the airline made a pre-tax profit of £75m ($141m) compared with £125m a year earlier. Rod Eddington, BA's chief executive, said the results were "respectable" in a third quarter when fuel costs rose by £106m or 47.3%. BA's profits were still better than market expectation of £59m, and it expects a rise in full-year revenues.
To help offset the increased price of aviation fuel, BA last year introduced a fuel surcharge for passengers.
In October, it increased this from £6 to £10 one-way for all long-haul flights, while the short-haul surcharge was raised from £2.50 to £4 a leg. Yet aviation analyst Mike Powell of Dresdner Kleinwort Wasserstein says BA's estim ...


## 1. Summarisation – Lead-3 vs TextRank vs DistilBART

In [3]:
from src.language_models.summarizer import IntelligentSummarizer
summ = IntelligentSummarizer(use_transformer=USE_TRANSFORMERS)
for method in ["lead", "textrank", "abstractive"]:
    r = summ.summarize_article(art.text, "balanced", method)
    print(f"--- {method} (used: {r['method']}, compression {r['compression']:.0%})"); print(r["summary"]); print()

20:49:15 | WARNING | src.language_models.summarizer | Abstractive model unavailable (No module named 'transformers') – using TextRank


--- lead (used: lead, compression 15%)
British Airways has blamed high fuel prices for a 40% drop in profits. Reporting its results for the three months to 31 December 2004, the airline made a pre-tax profit of £75m ($141m) compared with £125m a year earlier. Rod Eddington, BA's chief executive, said the results were "respectable" in a third quarter when fuel costs rose by £106m or 47.3%.

--- textrank (used: textrank, compression 17%)
Rod Eddington, BA's chief executive, said the results were "respectable" in a third quarter when fuel costs rose by £106m or 47.3%. Yet aviation analyst Mike Powell of Dresdner Kleinwort Wasserstein says BA's estimated annual surcharge revenues - £160m - will still be way short of its additional fuel costs - a predicted extra £250m. BA had previously forecast a 2% to 3% rise in full-year revenue.

--- abstractive (used: textrank, compression 17%)
Rod Eddington, BA's chief executive, said the results were "respectable" in a third quarter when fuel costs r

In [4]:
for kind in ["brief", "detailed"]:
    print(kind.upper(), "→", summ.summarize_article(art.text, kind)["summary"]); print()

BRIEF → Rod Eddington, BA's chief executive, said the results were "respectable" in a third quarter when fuel costs rose by £106m or 47.3%. BA had previously forecast a 2% to 3% rise in full-year revenue.

DETAILED → Rod Eddington, BA's chief executive, said the results were "respectable" in a third quarter when fuel costs rose by £106m or 47.3%. BA's profits were still better than market expectation of £59m, and it expects a rise in full-year revenues. Yet aviation analyst Mike Powell of Dresdner Kleinwort Wasserstein says BA's estimated annual surcharge revenues - £160m - will still be way short of its additional fuel costs - a predicted extra £250m. Turnover for the quarter was up 4.3% to £1.97bn, further benefiting from a rise in cargo revenue. BA had previously forecast a 2% to 3% rise in full-year revenue.



## 2. Headline generation – compared with the real BBC headline (ROUGE)

In [5]:
rows = []
for _, r in bbc.sample(100, random_state=0).iterrows():
    h = summ.generate_headlines(r.text)
    q = summ.assess_summary_quality(r.text, h["informative"], reference=r.title)
    rows.append({"real": r.title, "generated": h["informative"], **{k: q[k] for k in ("rouge1", "rougeL")}})
hd = pd.DataFrame(rows); print(hd[["rouge1", "rougeL"]].mean().round(3)); hd.head(6)

rouge1    0.220
rougeL    0.201
dtype: float64


,real,generated,rouge1,rougeL
0,Blair moves to woo Jewish voters,"Tony Blair has pledged to ""never",0.3333,0.3333
1,Kennedy questions trust of Blair,"Lib Dem leader Charles Kennedy has said voters now have a ""fundamental lack of",0.2105,0.2105
2,Brown calls for £5.5bn Aids fund,Gordon Brown has called on rich nations to fund a £5.5bn ($10bn) plan to,0.4545,0.3636
3,Berlin applauds Hotel Rwanda,Political thriller Hotel Rwanda was given a rousing reception by spectators at the Berlin,0.3333,0.2222
4,Britney attacks 'false tabloids',"Pop star Britney Spears has attacked ""false"" and ""desperate"" US tabloid magazines",0.5000,0.5000
5,Kennedy calls for Iraq exit plans,Tony Blair should set out a proper exit strategy from Iraq in the wake,0.2000,0.1000


## 3. Summary quality without references

In [6]:
from src.analysis.ner_extractor import EntityRelationshipMapper
mapper = EntityRelationshipMapper()
rows = []
for _, r in bbc.sample(80, random_state=1).iterrows():
    ents = list({e["canonical"] for e in mapper.extract_entities(r.text) if e["label"] in ("PERSON", "ORG", "GPE")})
    for m in ("lead", "textrank"):
        s = summ.summarize_article(r.text, "balanced", m)["summary"]
        rows.append({"method": m, **summ.assess_summary_quality(r.text, s, entities=ents or None)})
pd.DataFrame(rows).groupby("method").mean(numeric_only=True).round(3)

,compression,redundancy,novel_word_ratio,entity_coverage
method,,,,
lead,0.211,0.021,0.0,0.355
textrank,0.235,0.031,0.0,0.344


## 4. Multi-document summary (several articles about one story)

In [7]:
related = bbc[bbc.text.str.contains("Yukos")].text.tolist()
print(len(related), "articles mention Yukos")
print(summ.summarize_multiple_articles(related, n_sentences=5, focus_topic="Yukos oil company Russia"))

27 articles mention Yukos
State-owned Rosneft bought the Yugansk unit for $9.3bn in a sale forced by Russia to part settle a $27.5bn tax claim against Yukos. China lent Russia $6bn (£3.2bn) to help the Russian government renationalise the key Yuganskneftegas unit of oil group Yukos, it has been revealed. Russian oil firm Yukos has sued four companies for their role in last year's forced state auction of its key oil production unit Yuganskneftegas. The four companies named in the law suit are gas giant Gazprom, its unit Gazpromneft, investment company Baikal, and state oil firm Rosneft. A little-known Russian company has bought the main production unit of oil giant Yukos at auction in Moscow.


## 5. Semantic search
Hybrid score = 0.6 × dense similarity + 0.4 × TF-IDF match. With `sentence-transformers` the dense part is the
**multilingual** MiniLM model, so an English query also retrieves Spanish/German articles; offline it uses LSA.

In [8]:
from src.language_models.embeddings import SemanticEmbedder, SemanticSearchEngine, retrieval_eval
corpus = pd.concat([bbc, webz.sample(6000, random_state=0)], ignore_index=True)
engine = SemanticSearchEngine(SemanticEmbedder(use_transformer=USE_TRANSFORMERS)).encode_documents(corpus)
print("backend:", engine.embedder.backend)
engine.semantic_search("technology companies facing regulation", top_k=5)[["title", "language", "category", "score"]]

20:49:19 | WARNING | src.language_models.embeddings | Sentence-Transformer unavailable (No module named 'sentence_transformers') – falling back to LSA


20:49:59 | INFO    | src.language_models.embeddings | Indexed 8092 articles with lsa backend


backend: lsa


,title,language,category,score
7471,"Life science firms move ahead on AI, with concerns",en,tech,0.3319
4896,Could Anthropic's IPO Reshape the Global AI Market?,en,tech,0.3167
4756,The Future of Smart Technology: How Nvidia is Powering the Next Digital Revolution | Flipboard,en,tech,0.3130
210,Electronics firms eye plasma deal,en,business,0.3005
2600,Nvidia Partners With Major Financial Firms to Mobilize More Than $500 Billion for AI Infrastructure,en,tech,0.2929


In [9]:
engine.semantic_search("central bank raises interest rates", top_k=5,
                       filters={"language": ["es", "de", "fr"]})[["title", "language", "score"]]

,title,language,score
4610,Analystenprognosen zu Lindt&Sprüngli LISN (Tabelle) | Finanz und Wirtschaft,de,0.2598
3517,Geldautomat in Witten gesprengt,de,0.0965
4408,La Defensoría del Pueblo saluda la ratificación de Julio Velarde como presidente del Banco Central de la Reserva del...,es,0.0824
6990,Europa busca ampliar las subidas mientras el petróleo cae con fuerza | Mercados Financieros | Cinco Días,es,0.0751
6096,Máximo diplomático de China se reúne con asesor especial de presidente de Brasil_Spanish.china.org.cn_中国最权威的西班牙语新闻网站,es,0.0639


In [10]:
print("Query expansion:", engine.expand_query("ai regulation economy"))
print(retrieval_eval(engine, k=10, n_queries=200))

Query expansion: {'ai': ['artificial intelligence', 'machine learning', 'chatbot', 'generative'], 'regulation': ['rules', 'law', 'regulator', 'antitrust'], 'economy': ['economic', 'gdp', 'growth', 'inflation']}


{'k': 10, 'queries': 200, 'precision_at_k': 0.7925, 'backend': 'lsa'}


In [11]:
q = bbc.iloc[100]; print("QUERY ARTICLE:", q.title)
engine.find_similar_articles(q.title + ". " + q.text, top_k=5, exclude_doc_id=q.doc_id)[["title", "category", "similarity"]]

QUERY ARTICLE: Australia rates at four year high


,title,category,similarity
120,US interest rates increased to 2%,business,0.8254
2953,"Following the release of Australia's Consumer Inflation Expectations, AUD/USD trades near 0.6680, having declined - ...",business,0.8110
118,Bank opts to leave rates on hold,business,0.7922
428,ECB holds rates amid growth fears,business,0.7830
395,US interest rate rise expected,business,0.7781


In [12]:
engine.cluster_similar_content(n_clusters=8)

,cluster,size,top_terms,central_title
0,1,2350,"not, government, people, new, minister, election",An oath of his own: Stewart Rhodes' son takes GOP criticisms to the campaign trail
1,7,1417,"game, season, team, win, against, play",ECU GAMEDAY: Battle of 1-2 American teams sees Pirates prep for Owls' 'reboot' | Sports | reflector.com
2,2,1350,"new, not, technology, use, people, users",How to Implement a Winning SaaS Renewal Strategy
3,5,977,"gobierno, tras, contra, ante, tiene, así",Superventas francés muestra que la ciencia prueba la existencia de Dios - ZENIT - Espanol
4,0,974,"percent_value, currency_usd, market, growth, company, shares",Can Atlassian Stock Recover If Markets Fall?
5,6,429,"dell, qu, après, hanno, all, dopo",VIDEO - Quand le spectacle de drift tourne à la catastrophe
6,3,314,"pela, feira, sobre, aos, ainda, são","Após perder direito a último horário eleitoral, Engler não levará pedido de voto de Zema para TV"
7,4,281,"gegen, diese, unter, wir, können, beim",Nach Söder-Vorschlag: Bayerische Region will Unabhänbgigkeit


## 6. Content enhancement – grounded context for an article

In [13]:
from src.language_models.generator import ContentEnhancer
mapper.build_knowledge_graph(bbc.title + ". " + bbc.text.str[:2000], bbc.category)
enh = ContentEnhancer(mapper, engine)
target = bbc[bbc.title.str.contains("Blair")].iloc[0]
ents = mapper.extract_entities(target.text)
print(target.title)
enh.enhance_article(target.text, ents)

20:50:43 | INFO    | src.analysis.ner_extractor | Knowledge graph: 5009 nodes, 20624 edges


Blair buys copies of new Band Aid


{'entity_background': [{'entity': 'United States',
   'type': 'GPE',
   'mentions_in_corpus': 568,
   'mostly_in': 'business',
   'often_with': 'United Kingdom, Europe, New York'},
  {'entity': 'Tony Blair',
   'type': 'PERSON',
   'mentions_in_corpus': 128,
   'mostly_in': 'politics',
   'often_with': 'Labour, United Kingdom, Michael Howard'},
  {'entity': 'HMV',
   'type': 'ORG',
   'mentions_in_corpus': 4,
   'mostly_in': 'entertainment',
   'often_with': 'United Kingdom, Band Aid, Chris Martin'},
  {'entity': 'Band Aid',
   'type': 'ORG',
   'mentions_in_corpus': 5,
   'mostly_in': 'entertainment',
   'often_with': 'Chris Martin, Diana Ross, Lionel Richie'}],
 'related_articles': [{'doc_id': 'bbc_entertainment_159',
   'title': 'Band Aid 20 single storms to No 1',
   'similarity': 0.946},
  {'doc_id': 'bbc_entertainment_157',
   'title': 'US charity anthem is re-released',
   'similarity': 0.8058},
  {'doc_id': 'bbc_entertainment_160',
   'title': 'iTunes now selling Band Aid song'

In [14]:
enh.cross_reference_facts(target.text)

[]

## Key takeaways
* TextRank+MMR produces non-redundant summaries; DistilBART (Colab) writes fluent abstractive summaries – both behind one API.
* Generated headlines overlap meaningfully with the real ones (ROUGE), with zero training.
* Hybrid semantic search finds relevant articles by meaning, supports filters and expands queries; with multilingual
  embeddings it searches across languages.
* Enhancements are **grounded** in the system's own knowledge graph and corpus – no hallucinated facts.